# PipelineWatch-NG — Module 1: Data Ingestion
**Satellite-based crude oil theft monitoring · Niger Delta, Nigeria**

---

## Overview

Ingests three free satellite data streams over the Trans Niger Pipeline (TNP) corridor
using Google Earth Engine — **zero downloads, all processing server-side**.

| Sensor | Signal | Cloud-penetrating | Night-capable |
|--------|--------|:-----------------:|:-------------:|
| Sentinel-1 SAR | Oil spill dark spots | Yes — C-band radar | Yes — active sensor |
| FIRMS / VIIRS | Illegal refinery fire hotspots | Yes — thermal IR | Yes — thermal IR |
| TROPOMI SO2 | Chemical plumes from crude burning | Partial — UV | No — daytime only |

## Physics background

**SAR:** Oil on water dampens capillary waves, suppressing Bragg scattering of radar
pulses. Oil slicks appear as anomalously dark patches in VV polarisation imagery.

**VIIRS:** The 3.75 micron thermal IR band (T21) detects fire radiative power day and
night. Artisanal refineries burn continuously, producing persistent thermal anomalies
distinguishable from transient agricultural fires by their fixed location and high
detection count.

**TROPOMI SO2:** Artisanal crude cracking releases H2S which oxidises to SO2. TROPOMI
measures SO2 column density via UV backscatter. Persistent or episodic SO2 elevation
over a fixed location is the chemical fingerprint of illegal crude oil burning.

---

## Outputs

| File | Description |
|------|-------------|
| data/cached/sar_dark_spots.geojson | SAR dark spot sample points |
| data/cached/fire_hotspots.geojson | Persistent VIIRS fire hotspots |
| data/cached/so2_anomalies.geojson | TROPOMI SO2 anomaly zones |
| data/cached/m1_metadata.json | Run record |
| outputs/m1_fire_summary.html | Interactive fire summary chart |
| outputs/m1_so2_comparison.html | Interactive SO2 comparison chart |


## Cell 1 — Setup and GEE authentication

In [1]:
# Plotting backend
# matplotlib is set to Agg (non-interactive, file-output only) to prevent kernel
# crashes on Windows caused by the TkAgg GUI thread competing for memory.
# All plots are saved as interactive HTML via Plotly (outputs/) and as PNG via
# matplotlib for static use. To use interactive matplotlib on Linux/Mac,
# comment out the next two lines.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import IFrame, display

import ee
import os
import json
import gc
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime

# Replace with your GEE project ID (free at code.earthengine.google.com)
GEE_PROJECT_ID = 'pipelinewatch-ng'

try:
    ee.Initialize(project=GEE_PROJECT_ID)
    print('GEE connected  project: ' + GEE_PROJECT_ID)
    print('Connection test: ' + str(ee.Number(42).getInfo()) + '  (should be 42)')
except Exception:
    print('Not authenticated. Opening browser...')
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)
    print('Authenticated.')

CACHE_DIR  = '../data/cached'
OUTPUT_DIR = '../outputs'
os.makedirs(CACHE_DIR,  exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


C:\Users\taylo\anaconda3\envs\pipelinewatch\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


GEE connected  project: pipelinewatch-ng
Connection test: 42  (should be 42)


## Cell 2 — Study area configuration

In [2]:
ROI_BOUNDS     = [6.50, 5.00, 7.20, 5.80]   # [west, south, east, north]
ROI_CENTER     = [5.40, 6.85]
ROI_ZOOM       = 9
BASELINE_START = '2023-01-01'
BASELINE_END   = '2023-06-30'
RECENT_START   = '2024-01-01'
RECENT_END     = '2024-06-30'
SAR_DARK_SPOT_SIGMA = 1.5
FIRMS_BRIGHTNESS_K  = 330.0
SO2_THRESHOLD_DU    = 3.0
ROI = ee.Geometry.Rectangle(ROI_BOUNDS)
print('Study area : Trans Niger Pipeline corridor')
print('Baseline   : ' + BASELINE_START + ' to ' + BASELINE_END)
print('Recent     : ' + RECENT_START + ' to ' + RECENT_END)


Study area : Trans Niger Pipeline corridor
Baseline   : 2023-01-01 to 2023-06-30
Recent     : 2024-01-01 to 2024-06-30


## Cell 3 — Sentinel-1 SAR functions

In [3]:
def get_s1_collection(roi, start, end):
    return (
        ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterBounds(roi).filterDate(start, end)
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
        .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
        .select(['VV', 'VH'])
    )


def compute_sar_features(image, roi, sigma=1.5):
    # Note: dark_mag uses vv.subtract(mean_vv) — not mean_vv.subtract(vv) —
    # because the latter returns ee.Number which has no .rename() method.
    vv    = image.select('VV')
    vh    = image.select('VH')
    stats = vv.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=roi, scale=100, maxPixels=1e9, bestEffort=True)
    mean_vv   = ee.Number(stats.get('VV_mean'))
    std_vv    = ee.Number(stats.get('VV_stdDev'))
    threshold = mean_vv.subtract(std_vv.multiply(sigma))
    dark_mask = vv.lt(threshold).rename('dark_spot_mask')
    dark_mag  = vv.subtract(mean_vv).multiply(-1).divide(std_vv).rename('dark_spot_magnitude')
    vv_lin    = ee.Image(10).pow(vv.divide(10))
    vh_lin    = ee.Image(10).pow(vh.divide(10))
    ratio     = vv_lin.divide(vh_lin).rename('vv_vh_ratio')
    return image.addBands(dark_mask).addBands(dark_mag).addBands(ratio)


def build_s1_composite(roi, start, end, sigma=1.5):
    # Lee speckle filter is deferred to GEE Export tasks (not run interactively)
    # to avoid kernel memory exhaustion. Median compositing already suppresses
    # most transient noise (ships, rain, wind roughness).
    col   = get_s1_collection(roi, start, end)
    count = col.size().getInfo()
    print('  S1 scenes (' + start + ' to ' + end + '): ' + str(count))
    if count == 0:
        return None, None, 0
    col = col.map(lambda img: compute_sar_features(img, roi, sigma))
    return col.median().clip(roi), col, count


print('SAR functions defined.')


SAR functions defined.


## Cell 4 — FIRMS/VIIRS and TROPOMI functions

In [4]:
def get_firms_collection(roi, start, end):
    return (
        ee.ImageCollection('FIRMS').filterDate(start, end).filterBounds(roi)
        .select(['T21', 'confidence', 'line_number'])
    )


def compute_firms_composite(col, roi):
    return (
        col.select('T21').max().rename('T21_max').clip(roi)
        .addBands(col.select('T21').count().rename('fire_count').clip(roi))
        .addBands(col.select('T21').mean().rename('T21_mean').clip(roi))
    )


def extract_fire_hotspots(firms_comp, roi, t21_threshold_k=330.0, min_count=3):
    # .toInt() required before reduceToVectors — GEE requires integer bands
    hot_mask = (
        firms_comp.select('T21_max').gt(t21_threshold_k)
        .And(firms_comp.select('fire_count').gte(min_count))
    )
    vectors = (
        firms_comp.select('T21_max').updateMask(hot_mask).toInt()
        .reduceToVectors(geometry=roi, scale=375, geometryType='centroid',
                         eightConnected=True, maxPixels=1e9, bestEffort=True)
    )
    def annotate(f):
        geom  = f.geometry()
        stats = firms_comp.reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.max(), sharedInputs=True),
            geometry=geom.buffer(500), scale=375, maxPixels=1e6)
        t21_max = ee.Number(stats.get('T21_max_max'))
        count   = ee.Number(stats.get('fire_count_mean'))
        source  = ee.Algorithms.If(
            t21_max.gt(380), 'flare_candidate',
            ee.Algorithms.If(count.gte(5), 'refinery_candidate', 'agricultural_or_other'))
        return f.set({'T21_max_K': t21_max, 'T21_mean_K': ee.Number(stats.get('T21_mean_mean')),
                      'fire_count': count, 'signal_type': 'FIRMS_fire_hotspot',
                      'likely_source': source,
                      'confidence': ee.Algorithms.If(count.gte(10), 'high', 'nominal')})
    return vectors.map(annotate)


def get_tropomi_collection(roi, start, end):
    return (
        ee.ImageCollection('COPERNICUS/S5P/NRTI/L3_SO2')
        .filterDate(start, end).filterBounds(roi)
        .select(['SO2_column_number_density', 'cloud_fraction'])
    )


def compute_so2_composite(col, roi, max_cloud=0.3):
    # mol/m2 * 2241.5 = Dobson Units
    # cloud_fraction > 0.3 = retrieval unreliable, masked out
    def mask_and_convert(image):
        cloud  = image.select('cloud_fraction')
        so2_du = image.select('SO2_column_number_density').multiply(2241.5).rename('SO2_DU')
        return so2_du.updateMask(cloud.lt(max_cloud))
    clean = col.map(mask_and_convert)
    return (
        clean.mean().rename('SO2_mean_DU').clip(roi)
        .addBands(clean.max().rename('SO2_max_DU').clip(roi))
        .addBands(clean.reduce(ee.Reducer.percentile([75])).rename('SO2_p75_DU').clip(roi))
        .addBands(clean.count().rename('SO2_obs_count').clip(roi))
    )


def extract_so2_anomalies(so2_comp, roi, threshold_du=3.0, min_obs=5):
    # .multiply(100).toInt() scales small DU values before integer conversion
    # to preserve spatial boundary accuracy in reduceToVectors
    mask = (
        so2_comp.select('SO2_mean_DU').gt(threshold_du)
        .And(so2_comp.select('SO2_obs_count').gte(min_obs))
    )
    vectors = (
        so2_comp.select('SO2_mean_DU').updateMask(mask).multiply(100).toInt()
        .reduceToVectors(geometry=roi, scale=5500, geometryType='polygon',
                         eightConnected=True, maxPixels=1e9, bestEffort=True)
    )
    def annotate(f):
        geom     = f.geometry()
        stats    = so2_comp.reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.max(), sharedInputs=True),
            geometry=geom, scale=5500, maxPixels=1e6)
        so2_mean = ee.Number(stats.get('SO2_mean_DU_mean'))
        conf     = ee.Algorithms.If(so2_mean.gt(8), 'high',
                                    ee.Algorithms.If(so2_mean.gt(5), 'nominal', 'low'))
        return f.set({'SO2_mean_DU': so2_mean, 'SO2_max_DU': ee.Number(stats.get('SO2_max_DU_max')),
                      'obs_count': ee.Number(stats.get('SO2_obs_count_mean')),
                      'area_m2': geom.area(maxError=100),
                      'signal_type': 'TROPOMI_SO2_anomaly', 'confidence': conf})
    return vectors.map(annotate)


def compute_fire_gas_risk_score(so2_comp, firms_comp, roi,
                                 so2_threshold=2.0, t21_threshold=330.0):
    # 0=none  1=fire only  2=SO2 only  3=fire+SO2 co-located (high-confidence)
    so2_mask  = so2_comp.select('SO2_mean_DU').gt(so2_threshold)
    fire_mask = firms_comp.select('T21_max').gt(t21_threshold)
    return (
        so2_mask.reproject(crs='EPSG:4326', scale=5500).multiply(2)
        .add(fire_mask.reproject(crs='EPSG:4326', scale=5500))
        .rename('fire_gas_risk_score').clip(roi)
    )


print('FIRMS and TROPOMI functions defined.')


FIRMS and TROPOMI functions defined.


## Cell 5 — Run SAR ingestion

In [5]:
print('=== Sentinel-1 SAR ===')
s1_baseline, s1_baseline_col, n_baseline = build_s1_composite(
    ROI, BASELINE_START, BASELINE_END)
s1_recent, s1_recent_col, n_recent = build_s1_composite(
    ROI, RECENT_START, RECENT_END)
n_spots = 'deferred to Module 2'
print('Baseline scenes : ' + str(n_baseline))
print('Recent scenes   : ' + str(n_recent))


=== Sentinel-1 SAR ===
  S1 scenes (2023-01-01 to 2023-06-30): 30
  S1 scenes (2024-01-01 to 2024-06-30): 29
Baseline scenes : 30
Recent scenes   : 29


## Cell 6 — Run FIRMS ingestion

In [6]:
print('=== FIRMS / VIIRS ===')
firms_baseline_col  = get_firms_collection(ROI, BASELINE_START, BASELINE_END)
n_firms_base        = firms_baseline_col.size().getInfo()
firms_baseline_comp = compute_firms_composite(firms_baseline_col, ROI)
print('Baseline images: ' + str(n_firms_base))
firms_recent_col  = get_firms_collection(ROI, RECENT_START, RECENT_END)
n_firms_rec       = firms_recent_col.size().getInfo()
firms_recent_comp = compute_firms_composite(firms_recent_col, ROI)
print('Recent images  : ' + str(n_firms_rec))
fire_hotspots = extract_fire_hotspots(
    firms_recent_comp, ROI, t21_threshold_k=FIRMS_BRIGHTNESS_K, min_count=3)
n_hotspots = fire_hotspots.size().getInfo()
print('Persistent hotspots: ' + str(n_hotspots))
if n_hotspots > 0:
    for feat in fire_hotspots.limit(3).getInfo()['features']:
        p = feat['properties']
        print('  T21=' + str(round(p.get('T21_max_K', 0), 1)) + 'K'
              + '  count=' + str(round(p.get('fire_count', 0), 1))
              + '  source=' + str(p.get('likely_source', '?')))


=== FIRMS / VIIRS ===
Baseline images: 180
Recent images  : 181
Persistent hotspots: 50
  T21=349.3K  count=2.9  source=agricultural_or_other
  T21=336K  count=3.6  source=agricultural_or_other
  T21=338.2K  count=3  source=agricultural_or_other


## Cell 7 — Run TROPOMI ingestion

In [7]:
print('=== TROPOMI SO2 ===')
tropomi_base_col = get_tropomi_collection(ROI, BASELINE_START, BASELINE_END)
n_tropomi_base   = tropomi_base_col.size().getInfo()
so2_baseline     = compute_so2_composite(tropomi_base_col, ROI)
print('Baseline images: ' + str(n_tropomi_base))
tropomi_rec_col = get_tropomi_collection(ROI, RECENT_START, RECENT_END)
n_tropomi_rec   = tropomi_rec_col.size().getInfo()
so2_recent      = compute_so2_composite(tropomi_rec_col, ROI)
print('Recent images  : ' + str(n_tropomi_rec))
so2_anomalies = extract_so2_anomalies(so2_recent, ROI, threshold_du=SO2_THRESHOLD_DU, min_obs=5)
n_so2 = so2_anomalies.size().getInfo()
print('SO2 anomaly zones: ' + str(n_so2))
print('(Note: wet season cloud masking suppresses SO2 mean — see Module 2b for dry season)')
fire_gas_score = compute_fire_gas_risk_score(
    so2_recent, firms_recent_comp, ROI,
    so2_threshold=SO2_THRESHOLD_DU, t21_threshold=FIRMS_BRIGHTNESS_K)
print('Risk score ready.')


=== TROPOMI SO2 ===
Baseline images: 313
Recent images  : 278
SO2 anomaly zones: 0
(Note: wet season cloud masking suppresses SO2 mean — see Module 2b for dry season)
Risk score ready.


## Cell 8 — Summary table

In [8]:
df_summary = pd.DataFrame([
    {'Sensor': 'Sentinel-1 SAR', 'Scenes': n_recent,
     'Detections': n_spots, 'Type': 'Oil spill dark spots', 'All-weather': 'Yes', 'Night': 'Yes'},
    {'Sensor': 'FIRMS/VIIRS', 'Scenes': n_firms_rec,
     'Detections': n_hotspots, 'Type': 'Persistent fire hotspots', 'All-weather': 'Partial', 'Night': 'Yes'},
    {'Sensor': 'TROPOMI SO2', 'Scenes': n_tropomi_rec,
     'Detections': n_so2, 'Type': 'SO2 column anomalies', 'All-weather': 'Partial', 'Night': 'No'},
])
print('=== Module 1 Summary ===')
print(df_summary.to_string(index=False))
print('Generated: ' + datetime.now().strftime('%Y-%m-%d %H:%M'))


=== Module 1 Summary ===
        Sensor  Scenes           Detections                     Type All-weather Night
Sentinel-1 SAR      29 deferred to Module 2     Oil spill dark spots         Yes   Yes
   FIRMS/VIIRS     181                   50 Persistent fire hotspots     Partial   Yes
   TROPOMI SO2     278                    0     SO2 column anomalies     Partial    No
Generated: 2026-03-21 09:30


## Cell 9 — Fetch statistics for charts
**Important:** Data fetching (GEE) and plotting (Plotly/matplotlib) are always
kept in separate cells. Mixing GEE network calls with GUI rendering causes
kernel memory exhaustion on Windows.

In [9]:
# GEE calls only — no plotting in this cell
gc.collect()
base_fire = firms_baseline_comp.select('T21_max').reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.max(), sharedInputs=True),
    geometry=ROI, scale=375, maxPixels=1e9, bestEffort=True).getInfo()
gc.collect()
rec_fire = firms_recent_comp.select('T21_max').reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.max(), sharedInputs=True),
    geometry=ROI, scale=375, maxPixels=1e9, bestEffort=True).getInfo()
gc.collect()
base_so2 = so2_baseline.select('SO2_mean_DU').reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.max(), sharedInputs=True),
    geometry=ROI, scale=5500, maxPixels=1e9, bestEffort=True).getInfo()
gc.collect()
rec_so2 = so2_recent.select('SO2_mean_DU').reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.max(), sharedInputs=True),
    geometry=ROI, scale=5500, maxPixels=1e9, bestEffort=True).getInfo()

base_fire_mean = base_fire.get('T21_max_mean', 0) or 0
base_fire_max  = base_fire.get('T21_max_max',  0) or 0
rec_fire_mean  = rec_fire.get('T21_max_mean',  0) or 0
rec_fire_max   = rec_fire.get('T21_max_max',   0) or 0
base_so2_mean  = base_so2.get('SO2_mean_DU_mean', 0) or 0
rec_so2_mean   = rec_so2.get('SO2_mean_DU_mean',  0) or 0
print('Stats fetched. Run next cells to plot.')


Stats fetched. Run next cells to plot.


## Cell 10 — Fire summary chart (Plotly)

In [10]:
# Plotly only — no GEE calls in this cell
fig = go.Figure()
fig.add_trace(go.Bar(
    x=['Baseline mean', 'Recent mean', 'Baseline max', 'Recent max'],
    y=[base_fire_mean, rec_fire_mean, base_fire_max, rec_fire_max],
    marker_color=['#378ADD', '#E24B4A', '#85B7EB', '#F09595'],
    opacity=0.85,
    text=[str(round(v, 1)) for v in [base_fire_mean, rec_fire_mean, base_fire_max, rec_fire_max]],
    textposition='outside'
))
fig.add_hline(y=FIRMS_BRIGHTNESS_K, line_dash='dash', line_color='#854F0B',
              annotation_text='Fire threshold (' + str(FIRMS_BRIGHTNESS_K) + ' K)')
fig.update_layout(
    title='PipelineWatch-NG — VIIRS Fire Hotspot Summary<br>Baseline (2023) vs Recent (2024)',
    yaxis_title='T21 brightness temperature (K)', height=420,
    plot_bgcolor='white', paper_bgcolor='white')
fig.update_yaxes(gridcolor='#f0f0f0')
out = os.path.join(OUTPUT_DIR, 'm1_fire_summary.html')
fig.write_html(out)
print('Saved: ' + out)
display(IFrame(src=out, width='100%', height=450))


Saved: ../outputs\m1_fire_summary.html


## Cell 11 — SO2 comparison chart (Plotly)

In [11]:
# Plotly only — no GEE calls in this cell
fig = go.Figure()
fig.add_trace(go.Bar(
    x=['Baseline mean', 'Recent mean'],
    y=[base_so2_mean, rec_so2_mean],
    marker_color=['#378ADD', '#E24B4A'], opacity=0.85,
    text=[str(round(v, 4)) for v in [base_so2_mean, rec_so2_mean]],
    textposition='outside'
))
fig.add_hline(y=SO2_THRESHOLD_DU, line_dash='dash', line_color='#854F0B',
              annotation_text='Wet season threshold (' + str(SO2_THRESHOLD_DU) + ' DU)')
fig.update_layout(
    title='PipelineWatch-NG — TROPOMI SO2 Baseline vs Recent<br>'
          'Wet season cloud masking suppresses mean values — see Module 2b for dry season results',
    yaxis_title='SO2 column density (Dobson Units)', height=420,
    plot_bgcolor='white', paper_bgcolor='white')
fig.update_yaxes(gridcolor='#f0f0f0')
out = os.path.join(OUTPUT_DIR, 'm1_so2_comparison.html')
fig.write_html(out)
print('Saved: ' + out)
display(IFrame(src=out, width='100%', height=450))


Saved: ../outputs\m1_so2_comparison.html


## Cell 12 — Export GeoJSON outputs

In [12]:
def export_fc(fc, filename, label, max_features=500):
    path = os.path.join(CACHE_DIR, filename)
    n    = fc.size().getInfo()
    if n == 0:
        print('  ' + label + ': 0 features')
        with open(path, 'w') as f:
            json.dump({'type': 'FeatureCollection', 'features': []}, f)
        return 0
    print('  ' + label + ': ' + str(n) + ' features...')
    gj = fc.limit(max_features).getInfo()
    gj['metadata'] = {'exported': datetime.now().isoformat(), 'count': n,
                      'signal': label, 'period': RECENT_START + ' to ' + RECENT_END}
    with open(path, 'w') as f:
        json.dump(gj, f, indent=2)
    print('    ' + str(round(os.path.getsize(path)/1024, 1)) + ' KB')
    return n

print('=== Exporting GeoJSON ===')
sar_sampled = (
    s1_recent.select(['VV', 'dark_spot_mask', 'dark_spot_magnitude'])
    .sample(region=ROI, scale=200, numPixels=300, geometries=True, seed=42)
    .filter(ee.Filter.eq('dark_spot_mask', 1))
)
export_fc(sar_sampled,   'sar_dark_spots.geojson',  'SAR dark spots')
export_fc(fire_hotspots, 'fire_hotspots.geojson',   'VIIRS fire hotspots')
export_fc(so2_anomalies, 'so2_anomalies.geojson',   'TROPOMI SO2 anomalies')

meta = {'module': 'M1_ingestion', 'exported': datetime.now().isoformat(),
        'study_area': 'Trans Niger Pipeline corridor, Niger Delta',
        'bbox': ROI_BOUNDS,
        'scene_counts': {'s1_baseline': n_baseline, 's1_recent': n_recent,
                         'firms_baseline': n_firms_base, 'firms_recent': n_firms_rec,
                         'tropomi_baseline': n_tropomi_base, 'tropomi_recent': n_tropomi_rec},
        'detections': {'fire_hotspots': n_hotspots, 'so2_anomalies': n_so2}}
with open(os.path.join(CACHE_DIR, 'm1_metadata.json'), 'w') as f:
    json.dump(meta, f, indent=2)
print('Module 1 complete. Outputs in data/cached/ and outputs/')


=== Exporting GeoJSON ===
  SAR dark spots: 8 features...
    3.3 KB
  VIIRS fire hotspots: 50 features...
    28.6 KB
  TROPOMI SO2 anomalies: 0 features
Module 1 complete. Outputs in data/cached/ and outputs/


## Cell 13 — Interactive map (optional)

In [13]:
import geemap
m = geemap.Map(center=ROI_CENTER, zoom=ROI_ZOOM)
m.add_basemap('SATELLITE')
m.addLayer(s1_recent.select('VV'),
    {'min': -25, 'max': 0, 'palette': ['000000','404040','808080','c0c0c0','ffffff']},
    'S1 VV backscatter (dB)')
m.addLayer(s1_recent.select('dark_spot_mask').selfMask(),
    {'palette': ['E24B4A'], 'opacity': 0.7}, 'SAR dark spots — oil candidates')
m.addLayer(s1_baseline.select('dark_spot_mask').selfMask(),
    {'palette': ['378ADD'], 'opacity': 0.5}, 'SAR dark spots — BASELINE')
m.addLayer(firms_recent_comp.select('T21_max'),
    {'min': 300, 'max': 420, 'palette': ['E6F1FB','FAC775','EF9F27','E24B4A','791F1F']},
    'VIIRS max brightness temp (K)')
m.addLayer(so2_recent.select('SO2_mean_DU'),
    {'min': 0, 'max': 15, 'palette': ['E1F5EE','9FE1CB','1D9E75','085041','04342C']},
    'TROPOMI SO2 mean (Dobson Units)')
m.addLayer(fire_gas_score.selfMask(),
    {'min': 1, 'max': 3, 'palette': ['FAC775','EF9F27','E24B4A'], 'opacity': 0.75},
    'Combined risk score')
m.addLayer(ee.FeatureCollection([ee.Feature(ROI)]),
    {'color': '185FA5', 'fillColor': '00000000'}, 'TNP corridor')
m.add_layer_control()
m


Map(center=[5.4, 6.85], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', t…